<a href="https://colab.research.google.com/github/pandabo1985/agent_learn/blob/main/course/zh-CN/chapter3/section4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 一个完成的训练过程

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [18]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl

In [19]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# 1. 加载数据集
# 从 Hugging Face Hub 加载 GLUE 基准测试中的 MRPC（微软研究释义语料库）数据集
# 该任务用于判断两句话的意思是否相同（双句分类任务）
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")

# 2. 加载分词器（Tokenizer）
# 指定预训练模型的名称（这里使用的是基础版未区分大小写的 BERT 模型）
checkpoint = "bert-base-uncased"
# 自动下载并加载与该模型匹配的分词器，负责将文本转化为模型能听懂的数字
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


# 3. 定义分词预处理函数
def tokenize_function(example):
    # 将样本中的 'sentence1' 和 'sentence2' 同时传给分词器
    # 对于 BERT，它会自动将两句话拼接成：[CLS] 句子1 [SEP] 句子2 [SEP] 的格式
    # truncation=True 表示开启截断：如果拼接后的总长度超过模型最大限制（如512），自动切断超出的部分
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


# 4. 批量处理数据集
# 使用 .map() 遍历整个数据集，对所有样本执行上面定义的分词函数
# batched=True 表示开启批量处理，这样分词器可以利用多线程并行处理，极大提高分词速度
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
print(tokenized_datasets["train"].column_names)

# 5. 动态填充动态批处理器（Data Collator）
# 初始化一个数据收集器，它的作用是在组装每一个 Batch（批次）时，
# 自动将该 Batch 内所有样本通过 Padding（填充 0）对齐到当前 Batch 的“最长文本长度”，
# 这样既能节省显存，又能保证输入矩阵的形状整齐。
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']


In [20]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [21]:
["attention_mask", "input_ids", "labels", "token_type_ids"]

['attention_mask', 'input_ids', 'labels', 'token_type_ids']

In [23]:
from torch.utils.data import DataLoader

# 1. 构建训练集数据加载器
train_dataloader = DataLoader(
    tokenized_datasets["train"],  # 传入处理好的训练集数据（此时已是 PyTorch 张量格式）
    shuffle=True,  # 开启洗牌模式：每个训练轮次（Epoch）开始时，自动随机打乱数据顺序，防止模型产生记忆依赖
    batch_size=8,  # 批次大小：每次送入模型 8 条数据进行一次前向传播和反向传播
    collate_fn=data_collator,  # 指定批处理器：使用前面定义的 DataCollatorWithPadding，
    # 在组装这 8 条数据时，动态地把它们 Padding（填充 0）到这 8 条里最长的那句话的长度
)

# 2. 构建验证集数据加载器
eval_dataloader = DataLoader(
    tokenized_datasets["validation"],  # 传入处理好的验证集/测试集数据
    batch_size=8,  # 批次大小同样设为 8
    collate_fn=data_collator,  # 同样需要动态填充，确保同一个 Batch 内部的数据形状整齐
    # 注意：验证集不需要 shuffle=True，因为评估模型时按原始顺序读取即可，打乱顺序没有意义
)

In [25]:
import sys

# 假装系统中没有加载 torchvision，绕过 datasets 的版本检查 Bug
sys.modules.pop("torchvision", None)

for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 72]),
 'token_type_ids': torch.Size([8, 72]),
 'attention_mask': torch.Size([8, 72])}

In [14]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
# 1. 执行模型的前向传播（推理/预测）
# **batch 是 Python 的字典解包语法。它等价于：
# model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], ...)
# 模型内部会自动计算两句话的相似度，并因为传入了 'labels'，它还会顺便帮我们算好损失值（Loss）。
outputs = model(**batch)

# 2. 打印模型的评估结果
# outputs.loss: 打印当前批次的损失值（Tensor 标量）。训练的目标就是让这个 Loss 越来越小。
# outputs.logits.shape: 打印模型未归一化的预测得分矩阵形状。
print(outputs.loss, outputs.logits.shape)

tensor(0.7875, grad_fn=<NllLossBackward0>) torch.Size([8, 2])


In [28]:
from torch.optim import AdamW
# 初始化 AdamW 优化器
# model.parameters(): 告诉优化器需要更新该 BERT 模型内部的所有可学习参数（权重和偏置）
# lr=5e-5: 设置初始学习率（Learning Rate）为 0.00005。这是一个在微调 BERT 时非常经典的黄金学习率
optimizer = AdamW(model.parameters(), lr=5e-5)

In [29]:
from transformers import get_scheduler

# 1. 设置训练的总轮数（Epochs）
# 3 个 Epoch 意味着模型会将整个训练集完整地“学习” 3 遍
num_epochs = 3

# 2. 计算整个训练过程的总步数（Steps）
# 每一步（Step）代表模型吃掉一个 Batch 的数据并更新一次参数
# 总步数 = 总轮数 * 每个 Epoch 包含的 Batch 数量（即 DataLoader 的长度）
num_training_steps = num_epochs * len(train_dataloader)

# 3. 创建学习率调度器
lr_scheduler = get_scheduler(
    "linear",  # 指定调度器类型为 "linear"（线性衰减）：学习率会从初始值（如 5e-5）开始，像滑滑梯一样均匀、线性地降到 0
    optimizer=optimizer,  # 绑定前面创建的优化器，以便调度器可以直接修改优化器里的学习率
    num_warmup_steps=0,  # 预热步数（Warmup Steps）：设置为 0 代表不需要前期“热身”，从第一步开始就直接开始线性递减
    num_training_steps=num_training_steps,  # 告诉调度器整个训练一共要走多少步，以便它精准计算每一步该降温多少
)

# 4. 打印总步数
# 方便开发者直观地看到整个训练过程中，参数一共会被更新多少次
print(num_training_steps)

1377


In [30]:
import torch

# 1. 自动检测并指定计算设备
# torch.cuda.is_available(): 检查当前运行环境（如 Colab）是否分配了可用的英伟达 GPU
# 如果有 GPU，则设备设为 "cuda"；如果没有（比如用的是纯 CPU 环境），则无奈地降级设为 "cpu"
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# 2. 将模型搬运到指定的设备上
# 预训练模型默认是加载在 CPU 内存中的。
# 执行 model.to(device) 后，如果 device 是 cuda，模型的所有参数权重就会被“搬运”到显卡显存中，准备进行高速矩阵运算
model.to(device)

# 3. 打印当前正在使用的设备
# 在 Jupyter Notebook 中，单独写一行变量名会直接在下方输出结果，
# 方便你肉眼确认此时到底是用上了宝贵的 GPU（输出 device(type='cuda')），还是在用老旧的 CPU
device

device(type='cpu')

In [ ]:
from tqdm.auto import tqdm

# 1. 初始化进度条
# tqdm 可以在代码执行时在下方渲染出一个非常直观的、带百分比和剩余时间的进度条
# range(num_training_steps) 传入了总步数（1377步），这样进度条就知道走到哪里才是终点
progress_bar = tqdm(range(num_training_steps))

# 2. 将模型切换为训练模式
# 这行代码会启用 BERT 内部的 Dropout 等机制。在训练前必须显式调用！
model.train()

# 3. 外层循环：控制训练的轮数（Epoch）
for epoch in range(num_epochs):

    # 4. 内层循环：从数据加载器中批量读取数据（每一个 batch 包含 8 条数据）
    for batch in train_dataloader:

        # 5. 【极其关键】将该批次的所有数据搬运到相同的设备上（如 GPU）
        # 语法解读：利用字典推导式，遍历 batch 字典中的键值对，把所有的 Tensor 统一执行 .to(device)
        # 确保数据和模型都在同一个硬件（cuda）上，否则 PyTorch 会直接报错崩溃
        batch = {k: v.to(device) for k, v in batch.items()}

        # 6. 前向传播（Forward Pass）
        # 将这 8 条数据喂给模型，让模型进行预测，得到输出结果
        outputs = model(**batch)

        # 7. 获取当前批次的损失值（Loss）
        loss = outputs.loss

        # 8. 反向传播（Backward Pass）
        # 顺着损失值往回走，利用链式法则自动计算模型中所有可学习参数（权重和偏置）的梯度（Gradient）
        loss.backward()

        # 9. 参数更新（Optimization Step）
        # 优化器根据刚才计算出的梯度，正式动手修改并更新模型的内部参数，让预测越来越准
        optimizer.step()

        # 10. 更新学习率（Scheduler Step）
        # 学习率调度器开始干活，按照我们之前定好的“线性降温表”，把学习率稍微往下调一点点
        lr_scheduler.step()

        # 11. 梯度清零（Zero Gradients）
        # 【非常重要】在进入下一个 batch 之前，必须把刚刚计算的梯度清空
        # 因为 PyTorch 默认会累加梯度。如果不清零，下一个 batch 的梯度就会和上一次的混在一起，导致模型训练彻底崩溃
        optimizer.zero_grad()

        # 12. 进度条前进一步
        # 告诉 tqdm 已经成功完成了一步参数更新，下方可视化的进度条会向右走一格
        progress_bar.update(1)

  0%|          | 0/1377 [00:00<?, ?it/s]

In [ ]:
import evaluate

# 1. 加载评估指标计算器
# 使用 Hugging Face 的 evaluate 库，专门加载针对 GLUE 基准测试中 MRPC 任务的评估标准
# 它会自动帮我们准备好计算 Accuracy（准确率）和 F1-score（F1分数）的公式
metric = evaluate.load("nyu-mll/glue", "mrpc")

# 2. 将模型切换为评估/测试模式
# 这行代码会关闭 BERT 内部的 Dropout 等只在训练时有用的机制，确保测试时的预测结果稳定且可重复
model.eval()

# 3. 循环遍历验证集数据加载器
for batch in eval_dataloader:
    # 4. 将验证集的一个 batch 数据搬运到相同的设备上（如 GPU）
    batch = {k: v.to(device) for k, v in batch.items()}

    # 5. 【极其关键】关闭梯度计算上下文
    # with torch.no_grad() 告诉 PyTorch 此时我们只需要做前向推理，不需要记录任何梯度信息。
    # 这样可以大幅度减少内存/显存占用，并且加快预测速度（因为评估不需要反向传播更新参数）
    with torch.no_grad():
        outputs = model(**batch)

    # 6. 获取模型未归一化的预测得分（Logits）
    logits = outputs.logits

    # 7. 将得分转化为具体的预测标签
    # logits 的形状是 [8, 2]，代表 8 条数据里每条数据对“分类0”和“分类1”的得分
    # torch.argmax(..., dim=-1) 会找出每一行里得分最高的那个位置的索引（即 0 或 1），作为模型的最终预测答案
    predictions = torch.argmax(logits, dim=-1)

    # 8. 累计当前批次的测试结果
    # 将模型刚刚猜出来的答案（predictions）和真实的正确答案（references）一起塞给指标计算器缓存起来
    metric.add_batch(predictions=predictions, references=batch["labels"])

# 9. 汇总并计算最终的评估得分
# 当所有批次的数据都考完后，调用 .compute() 统一对刚才缓存的所有答案进行比对，
# 最终计算并返回整个验证集上的总准确率和 F1 分数
metric.compute()

{'accuracy': 0.8431372549019608, 'f1': 0.8907849829351535}

In [ ]:
import torch
from torch.optim import AdamW  # 推荐：改为从 torch.optim 导入以避免旧版 transformers 的导入报错
from transformers import AutoModelForSequenceClassification, get_scheduler

# 1. 加载用于序列分类的预训练模型
# checkpoint 在前面定义过（如 "bert-base-uncased"）
# num_labels=2 表示这是一个二分类任务（在 MRPC 中代表：两句话意思相同 / 不同）
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=2
)

# 2. 初始化 AdamW 优化器
# model.parameters() 告诉优化器需要更新模型内部的所有权重和偏置
# lr=3e-5 (0.00003) 是微调 BERT 大模型时非常经典的黄金学习率，防止破坏预训练阶段学到的通用知识
optimizer = AdamW(model.parameters(), lr=3e-5)

# 3. 配置硬件加速设备 (GPU / CPU)
# 自动检测当前环境是否支持 GPU 加速。如果有则使用 "cuda"，没有则无奈降级到普通的 "cpu"
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# 将模型的所有参数权重一次性搬运到指定的设备中（如果是 cuda，则送入显卡显存）
model.to(device)

# 4. 配置学习率降温调度器（Scheduler）
num_epochs = 3  # 模型将整个训练集完整地“学习” 3 遍
# 计算总训练步数 = 总轮数 * 每个 Epoch 包含的批次数量（即 DataLoader 的长度）
num_training_steps = num_epochs * len(train_dataloader)

lr_scheduler = get_scheduler(
    "linear",  # 指定为 "linear" 线性衰减：学习率会随时间像滑滑梯一样均匀降到 0
    optimizer=optimizer,  # 绑定优化器
    num_warmup_steps=0,  # 预热步数为 0，代表从第一步开始直接线性递减
    num_training_steps=num_training_steps,  # 告诉调度器总步数（1377步），以便它精准分配每一步的降温幅度
)

# 5. 初始化进度条
# tqdm 可以在代码格下方渲染出一个带百分比和剩余时间的直观进度条
progress_bar = tqdm(range(num_training_steps))

# 6. 正式进入模型训练大循环
model.train()  # 【重要】将模型切换为训练模式，激活 Dropout 等训练机制
for epoch in range(num_epochs):  # 外层循环：控制 Epoch 轮数
    for batch in train_dataloader:  # 内层循环：每次从数据卡车里读取 8 条数据（Batch Size=8）
        # 【关键】利用字典推导式，把当前批次的所有 Tensor 矩阵全部搬运到相同的硬件设备（如 GPU）上
        batch = {k: v.to(device) for k, v in batch.items()}

        # 前向传播：把这 8 条数据喂给模型，模型进行预测并自动计算出当前批次的误差损失值（Loss）
        outputs = model(**batch)
        loss = outputs.loss

        # 反向传播：利用链式法则计算出模型所有参数相对于当前 Loss 的梯度（找出错在哪里）
        loss.backward()

        # 参数更新：优化器根据刚才计算出的梯度，正式动手修改并优化模型内部的权重
        optimizer.step()

        # 更新学习率：调度器干活，让学习率平滑地减小一点点
        lr_scheduler.step()

        # 梯度清零：【极其重要】把刚才草稿纸上的梯度擦掉，防止它们和下一个 Batch 的梯度混在一起
        optimizer.zero_grad()

        # 进度条走一格，实时更新显示界面
        progress_bar.update(1)

In [ ]:
import torch
from accelerate import Accelerator
from torch.optim import AdamW  # 推荐：改为从 torch.optim 导入，防止旧版 transformers 报错
from transformers import AutoModelForSequenceClassification, get_scheduler

# 1. 初始化加速器（Accelerator）
# 它是整个高级训练流水线的“总指挥”。它会自动检测你的硬件环境（单卡、多卡、TPU还是混合精度FP16/BF16），
# 并在幕后自动帮你配置好所有的分布式计算环境。
accelerator = Accelerator()

# 2. 加载模型与配置常规优化器
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=2
)
optimizer = AdamW(model.parameters(), lr=3e-5)

# 3. 【最核心步骤】使用加速器一键打包/准备（Prepare）所有核心组件
# 语法解读：把你的数据加载器、模型和优化器全部扔进 accelerator.prepare() 中。
# 它在幕后默默帮你做了几件大事：
#   A. 自动把模型和数据放到对的设备上（相当于自动帮你执行了 model.to(device)，代码中再也不需要手动写了！）
#   B. 如果是多卡或分布式训练，自动把 Dataloader 包装成多卡并行的分布式数据流（DistributedDataLoader）
train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)

# 4. 计算训练总步数
num_epochs = 3
# 注意：这里必须使用被 accelerator 包装过后的 train_dl 来计算长度！
# 因为在多卡分布式训练下，数据会被平分给各个显卡，train_dl 的 batch 数量会相应减少。
num_training_steps = num_epochs * len(train_dl)

# 5. 配置学习率降温调度器
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

# 6. 初始化进度条
progress_bar = tqdm(range(num_training_steps))

# 7. 正式进入智能化分布式训练循环
model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        # ✨【巨大变化 1】：这里再也不需要写像刚才那样的 batch = {k: v.to(device) ...} 了！
        # 因为 train_dl 已经被 accelerator 托管，倒出来的 batch 已经在对应的 GPU/TPU 显存中了。

        # 前向传播：模型直接在对应硬件中进行推理和算 Loss
        outputs = model(**batch)
        loss = outputs.loss

        # ✨【巨大变化 2】将原生的 loss.backward() 替换为加速器的反向传播
        # 这行代码在多卡/多机训练时至关重要。它不仅计算梯度，还会自动跨多张显卡进行梯度同步（Gradient All-Reduce）。
        accelerator.backward(loss)

        # 参数更新、学习率递减、梯度清零
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        # 进度条前进
        progress_bar.update(1)

In [ ]:
from accelerate import notebook_launcher

# notebook_launcher 的作用：
# 1. 它会作为一个“启动器”，在后台帮你初始化多进程环境（Multiprocessing）。
# 2. 它要求你把所有的训练逻辑（模型加载、数据处理、训练循环）都封装在一个函数里（这里是 training_function）。
# 3. 它会自动处理 Jupyter Notebook 和底层多进程启动之间的兼容性，
#    确保即使在 Notebook 里，你的代码也能利用多块显卡或者 TPU 进行高效并行训练。
# 4. 这里的参数 (training_function) 就是你定义的包含了整个训练逻辑的函数名。
notebook_launcher(training_function)